# AgriPV optimizer v7 (vertical bifacial fences) — NSGA-II + representative irradiance sampling

- NSGA-II remains the multi-objective optimizer for LER and annual net income per hectare.
- Representative-hour weighting refers only to irradiance reconstruction: target-hour GHI / sampled-hour GHI.
- Representative-day, combined, and full-hour verification modes remain available.
- The uploaded 2020–2025 DE/LU hourly electricity-price tables are used by default.
- Full-year PV, one actual wheat season; both constructions are retained and optimized separately.
- v7: two notebooks share one engine: agripv_optimizer_fixed.ipynb (fixed tilt, identical to v6.5.1 incl. cache keys) and agripv_optimizer_single_axis.ipynb (Tianze's single-axis tracker; results in results/single_axis_*) and agripv_optimizer_vertical.ipynb (Tianze's vertical bifacial fence; results in results/vertical_*). This notebook: vertical bifacial fences.
- v6.5.1: ground grid 0.5 m spacing; field depth and modules per row are explicit inputs (FIELD_PRESET small/large/custom).
- v6.5: Stage-2 re-check of the Pareto designs with denser sampling (Section F); default prices 2023-2025; physics cache key only tracks code that shapes the Radiance run.
- v6.4: fixed field, number of rows derived from the pitch (FIELD_PRESET "small"/"large" in Section A2); electricity of one module scaled to the field; prices no longer enter the physics cache key.


## Section A -- Tianze's radiation model (fixed hourly ground output)

In [ ]:
# Cell 1: Imports, external tools, and project folders

# Standard-library imports.
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4
import os
import subprocess
import sys

# Third-party imports.
import numpy as np
import pandas as pd
import pvlib
import bifacial_radiance as br


# Add the executables used by bifacial_radiance to the current environment.
def configure_external_tools() -> None:
    bundled_data = Path(br.__file__).resolve().parent / "data"
    radiance_roots = (
        Path("/usr/local/radiance"),
        Path("/opt/homebrew/opt/radiance"),
        Path.home() / "Radiance",
    )
    native_bins = [root / "bin" for root in radiance_roots if (root / "bin").exists()]

    # On macOS, prefer the installed native executable over bundled Linux binaries.
    executable_folders = native_bins if sys.platform == "darwin" else [bundled_data, *native_bins]
    os.environ["PATH"] = os.pathsep.join(
        [*(str(folder) for folder in executable_folders), os.environ.get("PATH", "")]
    )

    radiance_libraries = [
        root / "lib"
        for root in radiance_roots
        if (root / "lib").exists()
    ]
    if radiance_libraries:
        os.environ["RAYPATH"] = os.pathsep.join(
            [*(str(folder) for folder in radiance_libraries), "."]
        )


# Resolve this cross-platform project layout:
# AgriPV/notebooks/this_notebook.ipynb
# AgriPV/weather/user_weather_file.epw
# AgriPV/results/Linrui_fixed_tilt_scenarios.csv
# AgriPV/results/Hao_fixed_tilt_hourly_scenarios.csv
WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = (
    WORKING_DIRECTORY.parent
    if WORKING_DIRECTORY.name.lower() == "notebooks"
    else WORKING_DIRECTORY
)
WEATHER_DIRECTORY = PROJECT_ROOT / "weather"
RESULTS_DIRECTORY = PROJECT_ROOT / "results"
LINRUI_RESULTS_FILE = RESULTS_DIRECTORY / "Linrui_fixed_tilt_scenarios.csv"
HAO_RESULTS_FILE = RESULTS_DIRECTORY / "Hao_fixed_tilt_hourly_scenarios.csv"
configure_external_tools()

In [ ]:
# Cell 2: Input structures used by the comparison model

# Human-readable scenario label, crop, cultivation period, shared weather, and albedo.
@dataclass(frozen=True)
class CultivationScenario:
    scenario_name: str
    crop_name: str
    start_date: str
    end_date: str
    weather_filename: str
    albedo: float
    ground_grid_x: int
    ground_grid_y: int
    weather_time_basis: str = "pvgis_utc"


# Shared envelope, semi-transparent cells/glass, and independent bifacialities.
@dataclass(frozen=True)
class ModuleSpecification:
    name: str
    module_x: float
    module_y: float
    module_thickness: float
    numcellsx: int
    numcellsy: int
    xcell: float
    ycell: float
    bifaciality: float
    conventional_bifaciality: float
    glass_total_transmittance: float
    glass_specular_reflectance: float
    glass_haze: float


# Number of modules in each row and number of rows, shared by both constructions.
# field_depth_m > 0 fixes the field: n_rows is then derived for every design as
# floor(field_depth_m / pitch) and the ground/crop plot is n_mods wide and field_depth_m deep.
# field_depth_m == 0 keeps the old behaviour (n_rows rows, plot = n_rows * pitch deep).
@dataclass(frozen=True)
class ArraySpecification:
    n_mods: int
    n_rows: int
    field_depth_m: float = 0.0


# Variables supplied by Hao's optimizer.
@dataclass(frozen=True)
class PVDesign:
    tilt: float
    pitch: float
    azimuth: float
    clearance_height: float
    xcellgap: float
    ycellgap: float
    xgap_conventional: float


# Single-axis tracker variables (Tianze's single_axis.ipynb). axis_azimuth is the bearing of
# the torque tube; hub_height is the rotation-axis height; GCR = module_y / pitch is derived.
@dataclass(frozen=True)
class TrackingDesign:
    pitch: float
    axis_azimuth: float
    hub_height: float
    limit_angle: float
    backtrack: bool
    xcellgap: float
    ycellgap: float
    xgap_conventional: float

    @property
    def azimuth(self) -> float:
        # Orientation of the rows as a fixed-tilt "facing" azimuth (axis - 90 deg). Used ONLY to
        # orient the fixed ground/crop plot along the rows; the modules rotate every hour.
        return (self.axis_azimuth - 90.0) % 360.0


# Vertical bifacial fence variables (Tianze's vertical_pv.ipynb). Tilt is fixed at 90 deg.
# clearance_height = lower module edge above ground; Tianze used 0, here it is an input/decision
# variable (> ground sensor height) so that no ground sensor lies inside a panel.
from dataclasses import field as _dataclass_field


@dataclass(frozen=True)
class VerticalPVDesign:
    pitch: float
    azimuth: float
    clearance_height: float
    xcellgap: float
    ycellgap: float
    xgap_conventional: float
    tilt: float = _dataclass_field(default=90.0, init=False)


# Numerical and site settings kept constant within one optimization run.
@dataclass(frozen=True)
class ModelSettings:
    xgap_semi_transparent: float = 0.0
    ground_border_m: float = 1.0
    ground_sensor_height: float = 0.05
    conventional_surface_offset: float = 0.005
    accuracy: str = "low"
    hpc: bool = True
    # "all": ray-trace every module of the array for every sky (reference behaviour).
    # "center": trace only the interior modules of the middle row and scale to the full array.
    #           With n_mods == 3 this is exactly ONE module (middle row, middle module).
    pv_trace_mode: str = "all"
    n_workers: int = max(1, (os.cpu_count() or 1) // 2)


# Minimal handles needed while one of the two temporary scenes is running.
@dataclass
class SimulationContext:
    run_id: str
    pv_type: str
    module_xgap: float
    simulation_name: str
    scene: object
    total_octfile: str


SETTINGS = ModelSettings()

In [ ]:
# Cell 3: Inputs edited by Linrui and Hao

# Input labels used below:
# BOTH/physics = affects both irradiation calculations.
# BOTH/record = identifies both CSV rows but does not change the ray tracing.
# HAO/output or LINRUI/physics = primarily owned by that named output.
# Every field must still keep its stated Python type. For an unused numeric field,
# use the suggested valid placeholder; never enter arbitrary letters or symbols.

# LINRUI/output switch; Hao may leave False. Both switches False runs nothing.
GET_LINRUI_GROUND_IRRADIATION = False
# HAO/output switch; Linrui may leave False.
GET_HAO_ARRAY_RESULTS = True

# Put this EPW file inside AgriPV/weather/ and enter only its file name here.
# This Aachen EPW matches PVGIS 5.3 TMY built from 2005-2023 candidates.
# Reuse typical weather on the actual dates below; this is NOT a weather forecast.
# Years now control the solar position. Date-only inputs include both full days.
# For a winter season, enter 2024-10-01 to 2025-03-30, for example.
# February 29 repeats February 28 weather but uses February 29 solar positions.
# Project rule: if the weather data changes, use a new file name; never replace
# an existing EPW with different contents under the same file name.
scenario = CultivationScenario(
    scenario_name="Aachen_Crop_A_case_001",  # BOTH/record: unique readable case label.
    crop_name="Crop_A",  # BOTH/record: retained for crop-model linkage; no ray-tracing effect.
    start_date="2024-10-01",  # BOTH/physics: first actual solar-calendar date (inclusive).
    end_date="2025-03-30",  # BOTH/physics: last actual solar-calendar date (inclusive).
    weather_filename="Aachen_PVGIS_TMY.epw",  # BOTH/physics: EPW file inside weather/.
    weather_time_basis="pvgis_utc",  # BOTH/physics: audited PVGIS UTC row order.
    # For ordinary EPW with genuine local-standard-time rows, use "local_standard".
    ground_grid_x=15,  # LINRUI/physics: positive integer; ground-grid columns.
    ground_grid_y=15,  # LINRUI/physics: positive integer; ground-grid rows; total sensors = x * y.
    albedo=0.20,  # BOTH/physics: ground short-wave reflectance, decimal 0 to 1.
)

# Enter the shared outer module dimensions and the semi-transparent cell layout.
# Enter each construction's rear/front response ratio independently, from 0 to 1.
# The 0.80 values below are examples, not verified module datasheet values.
# Set a construction's bifaciality to 0 for monofacial PV; its rear scan is skipped.
# Enter the three glass optical inputs from 0 to 1. Total transmittance plus
# specular reflectance should not exceed 1; haze splits the transmitted share.
# The conventional scene uses the outer dimensions, cell counts, and its own bifaciality; it ignores
# xcell, ycell, xcellgap, and ycellgap as physical geometry.
module_specification = ModuleSpecification(
    name="LONGi_LR5_72HGD",  # BOTH/record: module identifier; no direct ray-tracing effect.
    module_x=1.134,  # BOTH/physics: outer module width along a row, metres.
    module_y=2.278,  # BOTH/physics: outer module length across the slope, metres.
    module_thickness=0.030,  # BOTH/physics: module/glass envelope thickness, metres.
    numcellsx=6,  # BOTH/physics: cell columns and conventional integration columns.
    numcellsy=24,  # BOTH/physics: cell rows and conventional integration rows.
    xcell=0.182,  # BOTH/physics: semi-transparent cell width, metres.
    ycell=0.091,  # BOTH/physics: semi-transparent cell length, metres.
    bifaciality=0.80,  # HAO/output: semi-transparent rear/front response; no ground-light effect.
    conventional_bifaciality=0.80,  # HAO/output: conventional rear/front response; 0 = monofacial.
    glass_total_transmittance=0.88,  # LINRUI/physics: transmitted share; Hao may keep 0.88.
    glass_specular_reflectance=0.08,  # LINRUI/physics: reflected share; Hao may keep 0.08.
    glass_haze=0.2,  # LINRUI/physics: diffuse share, 0 to 1; Hao may keep 0.0 if haze is not varied.
)

# Enter the number of modules in each row and the number of rows once.
array_specification = ArraySpecification(
    n_mods=5,  # BOTH/physics: number of modules in each row, positive integer.
    n_rows=5,  # BOTH/physics: number of array rows, positive integer.
)

# The optimizer replaces these seven values for every paired comparison.
# xcellgap and ycellgap affect only the semi-transparent module.
# xgap_conventional affects only the gaps between conventional modules.
candidate_design = PVDesign(
    tilt=25.0,  # BOTH/physics: fixed module tilt from horizontal, degrees.
    pitch=7.0,  # BOTH/physics: centre-to-centre row spacing, metres.
    azimuth=180.0,  # BOTH/physics: module-facing azimuth, degrees.
    clearance_height=3.0,  # BOTH/physics: lower-edge clearance, metres.
    xcellgap=0.0004,  # BOTH/physics: semi-transparent horizontal cell gap, metres.
    ycellgap=0.00235,  # BOTH/physics: semi-transparent vertical cell gap, metres.
    xgap_conventional=0.20,  # BOTH/physics: gap between conventional modules, metres.
)

In [ ]:
# Cell 4: Geometry validation, variable cell grid, and glass margins

# Calculate the active-cell grid and the remaining glass margins.
def module_geometry(
    design: PVDesign,
    module_spec: ModuleSpecification,
) -> dict:
    grid_x = (
        module_spec.numcellsx * module_spec.xcell
        + (module_spec.numcellsx - 1) * design.xcellgap
    )
    grid_y = (
        module_spec.numcellsy * module_spec.ycell
        + (module_spec.numcellsy - 1) * design.ycellgap
    )

    return {
        "cell_module": {
            "numcellsx": module_spec.numcellsx,
            "numcellsy": module_spec.numcellsy,
            "xcell": module_spec.xcell,
            "ycell": module_spec.ycell,
            "xcellgap": design.xcellgap,
            "ycellgap": design.ycellgap,
        },
        "grid_x": grid_x,
        "grid_y": grid_y,
        "edge_x": (module_spec.module_x - grid_x) / 2,
        "edge_y": (module_spec.module_y - grid_y) / 2,
    }


# Stop before Radiance starts if the physical cells do not fit inside the module.
def validate_cell_array_fit(
    design: PVDesign,
    module_spec: ModuleSpecification,
) -> None:
    geometry = module_geometry(design, module_spec)
    tolerance_m = 1.0e-9
    overflow_axes = [
        axis for axis in ("x", "y")
        if geometry[f"grid_{axis}"] > getattr(module_spec, f"module_{axis}") + tolerance_m
    ]

    if not overflow_axes:
        return

    direction = (
        "both x and y directions" if len(overflow_axes) == 2
        else f"the {overflow_axes[0]} direction"
    )

    print(f"ERROR: The semi-transparent cell array exceeds the module in {direction}.")
    for axis in overflow_axes:
        print(
            f"{axis} direction: cell array = {geometry[f'grid_{axis}']:.6f} m, "
            f"module = {getattr(module_spec, f'module_{axis}'):.6f} m."
        )
    print("Reduce the corresponding cell size, cell count, or cell gap in Cell 3.")
    raise ValueError(
        "Invalid semi-transparent module geometry; Radiance was not started "
        "and no result was appended to either CSV file."
    )


# Preserve actual calendar years and support winter periods across New Year.
def cultivation_period_bounds(scenario):
    # Dates now specify the actual solar-position years (2024 in the examples).
    # Date-only inputs include the complete first and last day.
    start = pd.Timestamp(scenario.start_date).normalize()
    end = pd.Timestamp(scenario.end_date).normalize()
    if start.tzinfo is not None or end.tzinfo is not None:
        raise ValueError("Use calendar dates without a timezone; the EPW supplies standard time.")
    if end < start and end.year == start.year:
        end = end.replace(year=end.year + 1)
    if end < start:
        raise ValueError("end_date must not precede start_date.")
    return start, end


# Fill the glass envelope outside the active-cell grid.
def glass_margin_text(
    module_name: str,
    glass_material_name: str,
    geometry: dict,
    module_spec: ModuleSpecification,
) -> str:
    # bifacial_radiance shifts an even number of cell columns by half one cell.
    centre_x = (
        module_spec.xcell / 2
        if module_spec.numcellsx % 2 == 0
        else 0.0
    )

    edge_x = geometry["edge_x"]
    edge_y = geometry["edge_y"]
    grid_x = geometry["grid_x"]
    grid_y = geometry["grid_y"]

    # Left and right strips span the full module length.
    strips = [
        (
            "left",
            edge_x,
            module_spec.module_y,
            centre_x - module_spec.module_x / 2,
            -module_spec.module_y / 2,
        ),
        (
            "right",
            edge_x,
            module_spec.module_y,
            centre_x + grid_x / 2,
            -module_spec.module_y / 2,
        ),
        # Lower and upper strips fill the remaining length around the cell grid.
        (
            "lower",
            grid_x,
            edge_y,
            centre_x - grid_x / 2,
            -module_spec.module_y / 2,
        ),
        (
            "upper",
            grid_x,
            edge_y,
            centre_x - grid_x / 2,
            grid_y / 2,
        ),
    ]

    return "\n".join(
        f"! genbox {glass_material_name} {module_name}_glass_{side} "
        f"{width:.6f} {length:.6f} {module_spec.module_thickness:.6f} "
        f"| xform -t {xstart:.6f} {ystart:.6f} "
        f"{-module_spec.module_thickness / 2:.6f}"
        for side, width, length, xstart, ystart in strips
    )


# Ground sensor resolution is user-selectable; never accept zero or fractional counts.
def validate_ground_grid(scenario: CultivationScenario) -> None:
    for name in ("ground_grid_x", "ground_grid_y"):
        value = getattr(scenario, name)
        if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < 1:
            raise ValueError(f"{name} must be a positive integer in Cell 3.")


In [ ]:
# Cell 5: Shared actual-year weather, hourly solar skies, and module materials

def add_hazy_glass_material(
    demo: br.RadianceObj,
    module_spec: ModuleSpecification,
    material_name: str,
) -> None:
    # Total transmission and specular reflection come from ModuleSpecification.
    # Haze controls the regular-versus-diffuse split of transmitted light.
    nonabsorbed_colour = (
        module_spec.glass_total_transmittance
        / (1.0 - module_spec.glass_specular_reflectance)
    )
    transmitted_specular_fraction = 1.0 - module_spec.glass_haze
    material_text = (
        f"void trans {material_name}\n"
        "0\n"
        "0\n"
        f"7 {nonabsorbed_colour:.8f} {nonabsorbed_colour:.8f} "
        f"{nonabsorbed_colour:.8f} {module_spec.glass_specular_reflectance:.8f} "
        f"0.0 1.0 {transmitted_specular_fraction:.8f}\n"
    )
    material_file = Path("materials") / f"{material_name}.rad"
    material_file.write_text(material_text, encoding="ascii")
    demo.materialfiles.append(str(material_file))


def read_hourly_cultivation_weather(demo, scenario):
    # A non-leap reference is used ONLY to index the 8760 source weather records.
    # Solar positions are calculated separately on the actual requested dates.
    weather_path = WEATHER_DIRECTORY / scenario.weather_filename
    raw, metadata = demo._readEPW(
        epwfile=str(weather_path), label="right", coerce_year=2001,
    )
    epw_rows = pd.read_csv(weather_path, skiprows=8, header=None)
    if len(epw_rows) != len(raw):
        raise ValueError("EPW hourly row count mismatch")
    for column, number in {"_crop_temp": 6, "_crop_dew": 7, "_crop_wind": 21,
                           "_crop_rain": 33, "_crop_rain_period": 34}.items():
        raw[column] = pd.to_numeric(epw_rows[number], errors="raise").to_numpy()
    with weather_path.open(encoding="utf-8-sig") as handle:
        header = [next(handle) for _ in range(8)]
    offset_line = next((line for line in header if "Irradiance Time Offset (h):" in line), None)
    offset_h = float(offset_line.split(":", 1)[1]) if offset_line else -0.5

    if scenario.weather_time_basis == "pvgis_utc":
        if offset_line is None:
            raise ValueError("pvgis_utc requires the PVGIS irradiance offset in COMMENTS 2.")
        # Audited PVGIS export: row hour 1 contains JSON 00:00 UTC values.
        # Interpret its row labels in UTC FIRST, then convert to EPW standard time.
        # Do not apply this reinterpretation to an ordinary local-time EPW.
        raw.index = raw.index.tz_localize(None).tz_localize("UTC").tz_convert(raw.index.tz)
    elif scenario.weather_time_basis != "local_standard":
        raise ValueError("weather_time_basis must be 'pvgis_utc' or 'local_standard'.")

    start, end = cultivation_period_bounds(scenario)
    hour_starts = pd.date_range(
        start, end + pd.Timedelta(days=1), freq="h", inclusive="left", tz=raw.index.tz,
    )
    # TMY recycling is by LOCAL month/day/hour, after the source time conversion.
    source_keys = pd.Index((raw.index - pd.Timedelta(hours=1)).strftime("%m-%d-%H"))
    target_keys = hour_starts.strftime("%m-%d-%H").to_numpy(copy=True)
    leap_mask = (hour_starts.month == 2) & (hour_starts.day == 29)
    # Explicit agreed convention: repeat Feb 28 weather, not its solar position.
    target_keys[leap_mask] = hour_starts[leap_mask].strftime("02-28-%H")
    if not source_keys.is_unique:
        raise ValueError("The source EPW must contain exactly one record per TMY hour.")
    positions = source_keys.get_indexer(target_keys)
    if (positions < 0).any():
        raise ValueError("The source EPW does not contain all required TMY month/day/hour records.")
    selected = raw.iloc[positions].copy()
    selected.index = hour_starts + pd.Timedelta(hours=1)
    # Both Hao and Linrui use the same actual-year irradiance sampling instants.
    sampling_times = selected.index + pd.Timedelta(hours=offset_h)
    solar = pvlib.solarposition.get_solarposition(
        sampling_times, metadata["latitude"], metadata["longitude"], metadata["altitude"],
    )
    # Reusing typical weather may put small historical DNI at a below-horizon
    # target-year instant. Suppress that nonphysical beam; retain measured DHI.
    below_horizon = solar["elevation"].to_numpy() <= 0
    suppressed_beam_hours = int(((selected["DNI"].to_numpy() > 0) & below_horizon).sum())
    selected.loc[below_horizon, "DNI"] = 0.0
    # GHI is re-derived for the target-year geometry. gendaylit consumes DNI/DHI.
    selected["GHI"] = selected["DHI"].to_numpy() + selected["DNI"].to_numpy() * np.maximum(
        0.0, np.sin(np.radians(solar["elevation"].to_numpy())),
    )
    global ALIGNED_HOURLY_WEATHER, ALIGNED_WEATHER_METADATA
    ALIGNED_HOURLY_WEATHER = selected.copy()
    ALIGNED_WEATHER_METADATA = dict(metadata)
    lit = selected["GHI"].to_numpy() > 0
    metdata = br.MetObj(selected, metadata, label="right") if lit.any() else None
    if metdata is not None:
        # Override the wrapper's default midpoint/sunrise adjustment.
        metdata.solpos = solar.iloc[np.flatnonzero(lit)].copy()
        metdata.sunrisesetdata["corrected_timestamp"] = pd.Series(
            sampling_times[lit], index=metdata.sunrisesetdata.index,
        )
    timing = {
        "weather_time_basis": scenario.weather_time_basis,
        "irradiance_offset_from_hour_end_h": offset_h,
        "leap_day_weather": "repeat_Feb28",
        "leap_day_reused_hours": int(leap_mask.sum()),
        "below_horizon_dni_suppressed_hours": suppressed_beam_hours,
        "solar_position_start_year": start.year,
        "weather_scenario": "TMY_representative_weather_not_forecast",
    }
    print(f"Solar calendar: {start.date()} to {end.date()}; TMY weather, not a forecast.")
    print(f"Weather clock: {scenario.weather_time_basis}; irradiance offset = {offset_h:g} h from hour end.")
    if leap_mask.any():
        print("February 29: reused February 28 weather; solar position uses the actual leap day.")
    if suppressed_beam_hours:
        print(f"Suppressed historical DNI below the target-year horizon: {suppressed_beam_hours} hours.")
    return hour_starts, sampling_times, metdata, timing


def make_hourly_sky(demo, metdata, timeindex):
    # Full-spectrum solar sky. Never infer a replacement sun angle from old GHI.
    sun_elevation = float(metdata.solpos.iloc[timeindex]["elevation"])
    if sun_elevation <= 0:
        # Diffuse twilight: isotropic upper sky, no below-horizon direct sun.
        # This is an explicit approximation for positive DHI at twilight.
        dhi = float(metdata.dhi[timeindex])
        albedo = float(demo.ground.ReflAvg[0])
        skyfile = Path("skies") / f"twilight_{timeindex}.rad"
        skyfile.write_text(
            f"void glow twilight_sky\n0\n0\n4 {dhi / np.pi} {dhi / np.pi} {dhi / np.pi} 0\n"
            "twilight_sky source sky\n0\n0\n4 0 0 1 180\n"
            f"void glow twilight_ground\n0\n0\n4 {albedo*dhi/np.pi} {albedo*dhi/np.pi} {albedo*dhi/np.pi} 0\n"
            "twilight_ground source ground\n0\n0\n4 0 0 -1 180\n"
            f"void plastic ground_mat\n0\n0\n5 {albedo} {albedo} {albedo} 0 0\n"
            "ground_mat ring groundplane\n0\n0\n8 0 0 -.01 0 0 1 0 100\n",
            encoding="ascii",
        )
        return str(skyfile)
    # The installed wrapper skips DHI=0; epsilon preserves a direct-only hour.
    original_dhi = metdata.dhi[timeindex]
    if original_dhi == 0 and metdata.dni[timeindex] > 0:
        metdata.dhi[timeindex] = 1.0e-6
    try:
        skyfile = demo.gendaylit(timeindex=timeindex, metdata=metdata)
    finally:
        metdata.dhi[timeindex] = original_dhi
    if skyfile is None:
        raise ValueError(f"No hourly sky generated for {metdata.datetime[timeindex]}.")
    return skyfile


def create_selected_module(
    demo: br.RadianceObj,
    design: PVDesign,
    module_spec: ModuleSpecification,
    pv_type: str,
    module_xgap: float,
    run_id: str,
):
    module_name = f"{module_spec.name}_{pv_type}_{run_id}"
    module_parameters = dict(
        name=module_name, x=module_spec.module_x, y=module_spec.module_y,
        z=module_spec.module_thickness, xgap=module_xgap, ygap=0.0, numpanels=1,
    )
    if pv_type == "semi_transparent":
        glass_material_name = f"agripv_hazy_glass_{run_id}"
        add_hazy_glass_material(demo, module_spec, glass_material_name)
        geometry = module_geometry(design, module_spec)
        module = demo.makeModule(
            **module_parameters,
            cellModule=geometry["cell_module"],
            glass=True,
            glassEdge=0.0,
            bifi=module_spec.bifaciality,
            customtext=glass_margin_text(
                module_name,
                glass_material_name,
                geometry,
                module_spec,
            ),
        )
        # Replace bifacial_radiance's stock clear glass with the requested optics.
        module.text = module.text.replace("stock_glass", glass_material_name)
        # Save the edited geometry directly: compileText would regenerate stock glass.
        Path(module.modulefile).write_text(module.text, encoding="ascii")
    else:
        # Opaque geometry: no glass, cells, cell gaps, or transparent regions.
        # Bifacial response is independent of optical transparency.
        module = demo.makeModule(
            **module_parameters,
            glass=False,
            modulematerial="black",
            bifi=module_spec.conventional_bifaciality,
        )

    # Preserve the same outer envelope while applying construction-specific xgap.
    module.x = module_spec.module_x
    module.y = module_spec.module_y
    module.scenex = module_spec.module_x + module_xgap
    module.sceney = module_spec.module_y
    return module


In [ ]:
# Cell 6: Dynamic front/rear surface grids for both constructions

# Expand one module-centre scan into a rectangular integration grid.
def make_rectangular_surface_scans(
    analysis: br.AnalysisObj,
    scene,
    module_number: int,
    row_number: int,
    design: PVDesign,
    module_spec: ModuleSpecification,
    first_x: float,
    first_y: float,
    spacing_x: float,
    spacing_y: float,
    surface_offset: float,
) -> tuple[dict, dict]:
    front_scan, back_scan = analysis.moduleAnalysis(
        scene,
        modWanted=module_number,
        rowWanted=row_number,
        sensorsx=1,
        sensorsy=1,
        frontsurfaceoffset=surface_offset,
        backsurfaceoffset=surface_offset,
    )

    # Rotate local module X/Y directions into global scene coordinates.
    azimuth = np.radians(design.azimuth)
    tilt = np.radians(design.tilt)
    x_axis = (-np.cos(azimuth), np.sin(azimuth), 0.0)
    y_axis = (
        -np.cos(tilt) * np.sin(azimuth),
        -np.cos(tilt) * np.cos(azimuth),
        np.sin(tilt),
    )

    # Move both surfaces to the first centre and add grid increments.
    for scan in (front_scan, back_scan):
        scan["xstart"] = float(scan["xstart"]) + first_x * x_axis[0] + first_y * y_axis[0]
        scan["ystart"] = float(scan["ystart"]) + first_x * x_axis[1] + first_y * y_axis[1]
        scan["zstart"] = float(scan["zstart"]) + first_y * y_axis[2]
        scan.update(
            {
                "sx_xinc": spacing_x * x_axis[0],
                "sx_yinc": spacing_x * x_axis[1],
                "sx_zinc": 0.0,
                "xinc": spacing_y * y_axis[0],
                "yinc": spacing_y * y_axis[1],
                "zinc": spacing_y * y_axis[2],
                "Nx": module_spec.numcellsx,
                "Ny": module_spec.numcellsy,
                "Nz": 1,
            }
        )

    return front_scan, back_scan


# Select physical cell centres or equal opaque patches, and return represented area.
def make_module_surface_scans(
    analysis: br.AnalysisObj,
    context: SimulationContext,
    module_number: int,
    row_number: int,
    design: PVDesign,
    module_spec: ModuleSpecification,
) -> tuple[dict, dict, float]:
    if context.pv_type == "semi_transparent":
        # One point represents one physical semi-transparent cell.
        geometry = module_geometry(design, module_spec)
        spacing_x = module_spec.xcell + design.xcellgap
        spacing_y = module_spec.ycell + design.ycellgap
        centre_x = module_spec.xcell / 2 if module_spec.numcellsx % 2 == 0 else 0.0
        first_x = -geometry["grid_x"] / 2 + centre_x + module_spec.xcell / 2
        first_y = -geometry["grid_y"] / 2 + module_spec.ycell / 2
        surface_offset = module_spec.module_thickness / 2 + 0.001
        sample_area_m2 = module_spec.xcell * module_spec.ycell
    else:
        # One point represents one equal patch of the complete opaque surface.
        spacing_x = module_spec.module_x / module_spec.numcellsx
        spacing_y = module_spec.module_y / module_spec.numcellsy
        first_x = -module_spec.module_x / 2 + spacing_x / 2
        first_y = -module_spec.module_y / 2 + spacing_y / 2
        surface_offset = SETTINGS.conventional_surface_offset
        sample_area_m2 = spacing_x * spacing_y

    front_scan, back_scan = make_rectangular_surface_scans(
        analysis,
        context.scene,
        module_number,
        row_number,
        design,
        module_spec,
        first_x,
        first_y,
        spacing_x,
        spacing_y,
        surface_offset,
    )
    return front_scan, back_scan, sample_area_m2

In [ ]:
# Cell 7: Shared hourly tracing; Hao hourly outputs and Linrui period sums

def analyze_module(
    position: tuple[int, int],
    context: SimulationContext,
    design: PVDesign,
    module_spec: ModuleSpecification,
) -> tuple[float, float]:
    row_number, module_number = position
    analysis = br.AnalysisObj(
        context.total_octfile,
        context.simulation_name,
        hpc=SETTINGS.hpc,
    )

    # The construction selects the scan geometry and represented area once.
    front_scan, back_scan, sample_area_m2 = make_module_surface_scans(
        analysis,
        context,
        module_number,
        row_number,
        design,
        module_spec,
    )
    analysis_name = (
        f"{context.pv_type}_r{row_number:02d}_m{module_number:02d}_{context.run_id}"
    )
    bifaciality = (
        module_spec.bifaciality
        if context.pv_type == "semi_transparent"
        else module_spec.conventional_bifaciality
    )
    trace_rear = bifaciality > 0.0
    if not trace_rear:
        # Either construction can be monofacial; skip its unnecessary rear scan.
        front = analysis.analysis(
            context.total_octfile,
            analysis_name,
            front_scan,
            accuracy=SETTINGS.accuracy,
        )
        back = None
    else:
        front, back = analysis.analysis(
            context.total_octfile,
            analysis_name,
            front_scan,
            back_scan,
            accuracy=SETTINGS.accuracy,
        )

    # Representative hourly irradiance [W/m2] times represented area gives incident power [W].
    front_power = float(np.asarray(front["Wm2"], dtype=float).sum() * sample_area_m2)
    rear_effective_power = (
        0.0
        if back is None
        else float(
            np.asarray(back["Wm2"], dtype=float).sum()
            * sample_area_m2
            * bifaciality
        )
    )
    return front_power, rear_effective_power


def analyze_array(
    context: SimulationContext,
    design: PVDesign,
    module_spec: ModuleSpecification,
    array_spec: ArraySpecification,
) -> tuple[float, float]:
    positions = [
        (row_number, module_number)
        for row_number in range(1, array_spec.n_rows + 1)
        for module_number in range(1, array_spec.n_mods + 1)
    ]
    total_modules = len(positions)
    if SETTINGS.pv_trace_mode == "center":
        # Interior modules of the middle row: no row-end or array-edge neighbours missing.
        # Their mean power per module is scaled to the full array below.
        # 3 modules per row -> one module (the middle one); n_mods < 3 -> the whole middle row.
        middle_row = (array_spec.n_rows + 1) // 2
        inner = [m for m in range(1, array_spec.n_mods + 1) if 1 < m < array_spec.n_mods]
        positions = [(middle_row, m) for m in (inner or range(1, array_spec.n_mods + 1))]
    elif SETTINGS.pv_trace_mode != "all":
        raise ValueError("SETTINGS.pv_trace_mode must be 'all' or 'center'.")
    array_scale = total_modules / len(positions)

    # Run different modules in parallel using half of the logical CPUs.
    def run_module(position: tuple[int, int]) -> tuple[float, float]:
        return analyze_module(position, context, design, module_spec)

    front_total = 0.0
    rear_total = 0.0
    with ThreadPoolExecutor(max_workers=SETTINGS.n_workers) as executor:
        for front_power, rear_power in executor.map(run_module, positions):
            front_total += front_power
            rear_total += rear_power

    return front_total * array_scale, rear_total * array_scale


@dataclass(frozen=True)
class GroundGrid:
    # WORLD-fixed coordinates, independent of this design's tilt/pitch/azimuth.
    # Pass the SAME GroundGrid instance to calculate_hourly_results(...) and to
    # Linrui's evaluate_crop_yield_for_hao(..., grid=...) -- the point_id/x_m/y_m
    # she validates against must be produced by this exact object, every call.
    x_min: float
    x_max: float
    y_min: float
    y_max: float
    nx: int
    ny: int
    z_m: float = 0.05

    def points(self) -> pd.DataFrame:
        dx = (self.x_max - self.x_min) / self.nx
        dy = (self.y_max - self.y_min) / self.ny
        # Outer nx, inner ny -- matches bifacial_radiance's _linePtsMakeDict order
        # and Linrui's bridge.GroundGrid.points() row order.
        return pd.DataFrame([
            dict(point_id=f"p{i:03d}_{j:03d}",
                 x_m=self.x_min + (i + 0.5) * dx,
                 y_m=self.y_min + (j + 0.5) * dy,
                 cell_area_m2=dx * dy)
            for i in range(self.nx) for j in range(self.ny)
        ])

    def scan(self) -> tuple[dict, float]:
        pts = self.points()
        dx = (self.x_max - self.x_min) / self.nx
        dy = (self.y_max - self.y_min) / self.ny
        scan = dict(
            xstart=float(pts.iloc[0].x_m), ystart=float(pts.iloc[0].y_m), zstart=self.z_m,
            sx_xinc=dx, sx_yinc=0.0, sx_zinc=0.0,
            xinc=0.0, yinc=dy, zinc=0.0,
            Nx=self.nx, Ny=self.ny, Nz=1, orient="0 0 -1",
        )
        return scan, dx * dy


def default_ground_grid(scenario: CultivationScenario, array_spec: ArraySpecification,
                         module_spec: ModuleSpecification) -> "GroundGrid":
    # A generous, DESIGN-INDEPENDENT box: sized from the array's own dimensions
    # plus headroom, so it comfortably covers every NSGA-II candidate without
    # having to change shape between evaluations (Linrui's validator requires
    # the exact same point_id/x_m/y_m set on every call for a given scenario_id).
    # This is a starting default -- Hao/Linrui should widen it if the decision
    # variable bounds (pitch, azimuth) in Cell 3 ever exceed this margin.
    half_x = array_spec.n_mods * (module_spec.module_x + 1.0) / 2 + SETTINGS.ground_border_m
    depth = array_spec.field_depth_m if array_spec.field_depth_m > 0 else array_spec.n_rows * 16.0  # 16 m: generous max row pitch
    half_y = depth / 2 + SETTINGS.ground_border_m
    return GroundGrid(x_min=-half_x, x_max=half_x, y_min=-half_y, y_max=half_y,
                       nx=scenario.ground_grid_x, ny=scenario.ground_grid_y)


def trace_direct_ground_irradiation(analysis, octfile, scan):
    # Hourly gendaylit contains an explicit sun, unlike the old cumulative glow.
    # Zero ambient bounces keeps its beam/regular transmission, excluding diffuse
    # sky, diffuse glass scattering and diffuse surface reflection. -dr 0 disables
    # secondary virtual-source reflections in this direct component.
    completed = subprocess.run(
        ["rtrace", "-i", "-ab", "0", "-av", "0", "0", "0", "-dr", "0",
         "-dt", "0", "-dj", "0", "-h", "-ov", octfile],
        input=analysis._linePtsMakeDict(scan), text=True, capture_output=True, check=True,
    )
    rgb = np.asarray([[float(value) for value in line.split()]
                      for line in completed.stdout.splitlines() if line.strip()], dtype=float)
    return rgb.mean(axis=1)


def analyze_ground_hour(octfile, simulation_name, scan, analysis_name, with_components=True):
    # Per-sensor-point W/m2 -- NOT summed across points and NOT area-multiplied.
    # Both the day/point accumulation below and Linrui's crop bridge need an
    # areal density (Wh/m2/day), never a single period-total absolute Wh figure.
    analysis = br.AnalysisObj(octfile, simulation_name, hpc=SETTINGS.hpc)
    full = analysis.analysis(octfile, analysis_name, scan, accuracy=SETTINGS.accuracy)
    total_Wm2 = np.asarray(full["Wm2"], dtype=float)  # shape (Nx*Ny,)
    if not with_components:
        # The crop bridge only needs TOTAL shortwave. Skipping the extra direct-only
        # rtrace pass saves one trace per sky; direct/diffuse are then NaN placeholders.
        nothing = np.full_like(total_Wm2, np.nan)
        return nothing, nothing, total_Wm2
    direct_Wm2 = trace_direct_ground_irradiation(analysis, octfile, scan)  # shape (Nx*Ny,)
    diffuse_residual_Wm2 = total_Wm2 - direct_Wm2
    return direct_Wm2, diffuse_residual_Wm2, total_Wm2


def build_ground_day_frame(ground_daily: dict, points: pd.DataFrame, pv_type: str,
                           with_components: bool = True) -> pd.DataFrame:
    # Tidy long format: one row per (day, point) -- exactly what Linrui's
    # validate_radiation() in bridge.py expects, per scenario_id, per construction.
    rows = []
    for day_key, arr in sorted(ground_daily.items()):
        for point_row, values in zip(points.itertuples(index=False), arr):
            record = dict(
                day=day_key.isoformat(),
                pv_type=pv_type,
                point_id=point_row.point_id,
                x_m=point_row.x_m,
                y_m=point_row.y_m,
                cell_area_m2=point_row.cell_area_m2,
            )
            if with_components:
                record["direct_Wh_m2_day"] = float(values[0])
                record["diffuse_residual_Wh_m2_day"] = float(values[1])
            record["total_Wh_m2_day"] = float(values[2])
            rows.append(record)
    return pd.DataFrame(rows)


def calculate_hourly_results(
    design, scenario, module_spec, array_spec, run_id, run_directory,
    calculate_hao, calculate_linrui, ground_grid=None, sampling_config=None,
    crop_dates=None, constructions_to_run=None, ground_components=True,
):
    import time
    from sampling_core import SamplingConfig, make_sampling_plan
    sampling_config = sampling_config or SamplingConfig(mode="full")
    crop_dates = set(crop_dates) if crop_dates is not None else None
    results, ground_summary, timing = None, {}, {}
    constructions = (("semi_transparent", SETTINGS.xgap_semi_transparent),
                     ("conventional", design.xgap_conventional))
    for pv_type, module_xgap in constructions:
        if constructions_to_run is not None and pv_type not in constructions_to_run:
            continue
        grid = ground_grid[pv_type] if isinstance(ground_grid, dict) else ground_grid
        grid = grid or default_ground_grid(scenario, array_spec, module_spec)
        ground_points, (ground_scan, _) = grid.points(), grid.scan()
        construction_directory = run_directory / pv_type
        construction_directory.mkdir(parents=True, exist_ok=True)
        demo = br.RadianceObj(name=f"{pv_type}_{run_id}", path=str(construction_directory), hpc=SETTINGS.hpc)
        demo.setGround(scenario.albedo)
        hour_starts, solar_times, metdata, timing = read_hourly_cultivation_weather(demo, scenario)
        weather = ALIGNED_HOURLY_WEATHER
        plan = make_sampling_plan(hour_starts, weather.GHI, weather.DHI, sampling_config)
        buckets = {key: group for key, group in plan.dropna(subset=["sample_hour"]).groupby("sample_hour")}
        if results is None:
            results = pd.DataFrame(index=hour_starts)
            results["hour_start"] = hour_starts.map(lambda t: t.isoformat())
            results["hour_end"] = (hour_starts + pd.Timedelta(hours=1)).map(lambda t: t.isoformat())
            results["duration_h"] = 1.0
            results["solar_timestamp"] = solar_times.map(lambda t: t.isoformat())
            results["sample_hour"] = plan.sample_hour.map(lambda t: t.isoformat() if pd.notna(t) else "").to_numpy()
            results["irradiance_scale"] = plan.scale.to_numpy()
        module = create_selected_module(demo, design, module_spec, pv_type, module_xgap, run_id)
        scene = demo.makeScene(module=module, sceneDict=dict(
            tilt=design.tilt, pitch=design.pitch, azimuth=design.azimuth,
            clearance_height=design.clearance_height, nMods=array_spec.n_mods, nRows=array_spec.n_rows),
            radname=f"{pv_type}_{run_id}")
        columns = [f"{pv_type}_array_{face}_effective_incident_energy_Wh_hour" for face in ("front", "rear")]
        results[columns] = 0.0
        # Include dark crop days explicitly; absence is never silently interpreted as zero.
        days = sorted(crop_dates if crop_dates is not None else set(hour_starts.date))
        ground_daily = {day: np.zeros((len(ground_points), 3)) for day in days}
        completed = 0
        spent = dict(sky=0., octree=0., pv=0., ground=0.)  # seconds, for the timing summary
        started = time.perf_counter()
        for timeindex, hour_end in enumerate([] if metdata is None else metdata.datetime):
            sample_hour = hour_end - pd.Timedelta(hours=1)
            if sample_hour not in buckets:
                continue
            bucket = buckets[sample_hour]
            tick = time.perf_counter()
            skyfile = make_hourly_sky(demo, metdata, timeindex)
            spent["sky"] += time.perf_counter() - tick
            tick = time.perf_counter()
            octfile = demo.makeOct(filelist=[*demo.materialfiles, skyfile, *demo._getradfiles([scene])],
                                  octname=f"{pv_type}_{timeindex}_{run_id}")
            spent["octree"] += time.perf_counter() - tick
            context = SimulationContext(run_id=f"{run_id}_{timeindex}", pv_type=pv_type,
                module_xgap=module_xgap, simulation_name=demo.name, scene=scene, total_octfile=octfile)
            if calculate_hao:
                tick = time.perf_counter()
                powers = np.asarray(analyze_array(context, design, module_spec, array_spec))
                spent["pv"] += time.perf_counter() - tick
                results.loc[pd.DatetimeIndex(bucket.target_hour), columns] = bucket.scale.to_numpy()[:,None]*powers[None,:]
            crop_bucket = bucket if crop_dates is None else bucket[bucket.target_day.isin(crop_dates)]
            if calculate_linrui and not crop_bucket.empty:
                tick = time.perf_counter()
                direct, diffuse, total = analyze_ground_hour(octfile, demo.name, ground_scan,
                                                            f"ground_{pv_type}_{timeindex}_{run_id}",
                                                            with_components=ground_components)
                spent["ground"] += time.perf_counter() - tick
                values = np.column_stack([direct, diffuse, total])
                for day, weights in crop_bucket.groupby("target_day"):
                    ground_daily[day] += values*float(weights.scale.sum())
            completed += 1
            if completed % 20 == 0 or completed == len(buckets):
                elapsed = time.perf_counter() - started
                print(f"{pv_type}: {completed}/{len(buckets)} sampled skies, {elapsed/60:.1f} min elapsed; "
                      "s/sky: " + ", ".join(f"{k} {v/completed:.1f}" for k, v in spent.items()), flush=True)
        if completed != len(buckets):
            raise RuntimeError("Some representative skies were absent from MetObj; do not accept incomplete reconstruction")
        if calculate_linrui:
            ground_summary[pv_type] = build_ground_day_frame(ground_daily, ground_points, pv_type,
                                                              with_components=ground_components)
        timing.update(sampled_skies_per_construction=completed, sampling_mode=sampling_config.mode,
                      reconstruction="sample irradiance ratio times target-hour GHI")
    if results is None:
        raise ValueError("No construction selected")
    return results.reset_index(drop=True) if calculate_hao else pd.DataFrame(), ground_summary, timing


In [ ]:
# Cell 7b: Single-axis tracking physics (Tianze's single_axis.ipynb, adapted to sampling)
# Separate names from the fixed-tilt functions above, so the fixed-tilt code and its
# cached results stay unchanged. Per sampled hour the tracker angle comes from pvlib
# (bifacial_radiance set1axis, optional backtracking); that hour's scene is built at
# exactly this angle (no angle bins) and the module sensors rotate with the tracker.
from math import atan2, cos, degrees, radians, sin


def tracking_min_clearance(design: TrackingDesign, module_spec: ModuleSpecification) -> float:
    # Lowest point of the rotating module at the rotation limit (Tianze's check).
    critical_angle = degrees(atan2(module_spec.module_y, module_spec.module_thickness))
    angle = min(design.limit_angle, critical_angle)
    half_extent = 0.5 * (module_spec.module_y * sin(radians(angle))
                         + module_spec.module_thickness * cos(radians(angle)))
    return design.hub_height - half_extent


def validate_tracking_design(design: TrackingDesign, module_spec: ModuleSpecification) -> dict:
    validate_cell_array_fit(design, module_spec)
    if design.pitch <= 0 or design.hub_height <= 0:
        raise ValueError("pitch and hub_height must be positive.")
    if not 0.0 < design.limit_angle <= 90.0:
        raise ValueError("limit_angle must be greater than 0 and at most 90 degrees.")
    gcr = module_spec.module_y / design.pitch
    if not 0.0 < gcr <= 1.0:
        raise ValueError("Derived GCR = module_y / pitch must be in (0, 1]; increase pitch.")
    if tracking_min_clearance(design, module_spec) <= SETTINGS.ground_sensor_height:
        raise ValueError("The tracker can intersect the ground/sensor plane; increase hub_height "
                         "or reduce limit_angle.")
    return {"gcr": gcr}


def make_tracking_rectangular_scans(analysis, scene, module_number, row_number, module_spec,
                                    first_x, first_y, spacing_x, spacing_y, surface_offset):
    front_scan, back_scan = analysis.moduleAnalysis(
        scene, modWanted=module_number, rowWanted=row_number, sensorsx=1, sensorsy=1,
        frontsurfaceoffset=surface_offset, backsurfaceoffset=surface_offset,
    )
    # Local axes from THIS hour's scene orientation: sensors rotate with the tracker.
    azimuth = radians(float(scene.sceneDict["azimuth"]))
    tilt = radians(float(scene.sceneDict["tilt"]))
    x_axis = (-cos(azimuth), sin(azimuth), 0.0)
    y_axis = (-cos(tilt) * sin(azimuth), -cos(tilt) * cos(azimuth), sin(tilt))
    for scan in (front_scan, back_scan):
        scan["xstart"] = float(scan["xstart"]) + first_x * x_axis[0] + first_y * y_axis[0]
        scan["ystart"] = float(scan["ystart"]) + first_x * x_axis[1] + first_y * y_axis[1]
        scan["zstart"] = float(scan["zstart"]) + first_y * y_axis[2]
        scan.update({"sx_xinc": spacing_x * x_axis[0], "sx_yinc": spacing_x * x_axis[1], "sx_zinc": 0.0,
                     "xinc": spacing_y * y_axis[0], "yinc": spacing_y * y_axis[1], "zinc": spacing_y * y_axis[2],
                     "Nx": module_spec.numcellsx, "Ny": module_spec.numcellsy, "Nz": 1})
    return front_scan, back_scan


def make_tracking_surface_scans(analysis, pv_type, scene, module_number, row_number, design, module_spec):
    if pv_type == "semi_transparent":
        geometry = module_geometry(design, module_spec)
        spacing_x = module_spec.xcell + design.xcellgap
        spacing_y = module_spec.ycell + design.ycellgap
        represented_y = module_spec.ycell
        centre_x = module_spec.xcell / 2 if module_spec.numcellsx % 2 == 0 else 0.0
        first_x = -geometry["grid_x"] / 2 + centre_x + module_spec.xcell / 2
        first_lower_edge = -geometry["grid_y"] / 2
        surface_offset = module_spec.module_thickness / 2 + 0.001
        sample_area_m2 = module_spec.xcell * module_spec.ycell / 2
    else:
        spacing_x = module_spec.module_x / module_spec.numcellsx
        spacing_y = module_spec.module_y / module_spec.numcellsy
        represented_y = spacing_y
        first_x = -module_spec.module_x / 2 + spacing_x / 2
        first_lower_edge = -module_spec.module_y / 2
        surface_offset = SETTINGS.conventional_surface_offset
        sample_area_m2 = spacing_x * spacing_y / 2
    front_scans, back_scans = [], []
    for first_y in (first_lower_edge + represented_y / 4, first_lower_edge + 3 * represented_y / 4):
        front, back = make_tracking_rectangular_scans(
            analysis, scene, module_number, row_number, module_spec,
            first_x, first_y, spacing_x, spacing_y, surface_offset)
        front_scans.append(front)
        back_scans.append(back)
    return front_scans, back_scans, sample_area_m2


def analyze_tracking_module(position, pv_type, run_id, simulation_name, scene, octfile, design, module_spec):
    row_number, module_number = position
    analysis = br.AnalysisObj(octfile, simulation_name, hpc=SETTINGS.hpc)
    front_scans, back_scans, sample_area_m2 = make_tracking_surface_scans(
        analysis, pv_type, scene, module_number, row_number, design, module_spec)
    bifaciality = module_spec.bifaciality if pv_type == "semi_transparent" else module_spec.conventional_bifaciality
    front_sum = rear_sum = 0.0
    for index, (front_scan, back_scan) in enumerate(zip(front_scans, back_scans), start=1):
        name = f"{pv_type}_r{row_number:02d}_m{module_number:02d}_y{index}_{run_id}"
        if bifaciality > 0.0:
            front, back = analysis.analysis(octfile, name, front_scan, back_scan, accuracy=SETTINGS.accuracy)
            rear_sum += float(np.asarray(back["Wm2"], dtype=float).sum())
        else:
            front = analysis.analysis(octfile, name, front_scan, accuracy=SETTINGS.accuracy)
        front_sum += float(np.asarray(front["Wm2"], dtype=float).sum())
    return front_sum * sample_area_m2, rear_sum * sample_area_m2 * bifaciality


def analyze_tracking_array(pv_type, run_id, simulation_name, scene, octfile, design, module_spec, array_spec):
    # Same trace scope as the fixed array (SETTINGS.pv_trace_mode "all" or "center").
    positions = [(r, m) for r in range(1, array_spec.n_rows + 1) for m in range(1, array_spec.n_mods + 1)]
    total_modules = len(positions)
    if SETTINGS.pv_trace_mode == "center":
        middle_row = (array_spec.n_rows + 1) // 2
        inner = [m for m in range(1, array_spec.n_mods + 1) if 1 < m < array_spec.n_mods]
        positions = [(middle_row, m) for m in (inner or range(1, array_spec.n_mods + 1))]
    elif SETTINGS.pv_trace_mode != "all":
        raise ValueError("SETTINGS.pv_trace_mode must be 'all' or 'center'.")
    scale = total_modules / len(positions)

    def run_module(position):
        return analyze_tracking_module(position, pv_type, run_id, simulation_name, scene, octfile, design, module_spec)

    front_total = rear_total = 0.0
    with ThreadPoolExecutor(max_workers=SETTINGS.n_workers) as executor:
        for front_power, rear_power in executor.map(run_module, positions):
            front_total += front_power
            rear_total += rear_power
    return front_total * scale, rear_total * scale


def calculate_hourly_results_tracking(
    design, scenario, module_spec, array_spec, run_id, run_directory,
    calculate_hao, calculate_linrui, ground_grid=None, sampling_config=None,
    crop_dates=None, constructions_to_run=None, ground_components=True,
):
    # Same interface, sampling and outputs as calculate_hourly_results (fixed tilt).
    import time
    from sampling_core import SamplingConfig, make_sampling_plan
    sampling_config = sampling_config or SamplingConfig(mode="full")
    crop_dates = set(crop_dates) if crop_dates is not None else None
    results, ground_summary, timing = None, {}, {}
    gcr = validate_tracking_design(design, module_spec)["gcr"]
    constructions = (("semi_transparent", SETTINGS.xgap_semi_transparent),
                     ("conventional", design.xgap_conventional))
    for pv_type, module_xgap in constructions:
        if constructions_to_run is not None and pv_type not in constructions_to_run:
            continue
        grid = ground_grid[pv_type] if isinstance(ground_grid, dict) else ground_grid
        if calculate_linrui and grid is None:
            raise ValueError("Single-axis crop runs need the fixed plot grid from the engine")
        ground_points, ground_scan = (grid.points(), grid.scan()[0]) if grid is not None else (pd.DataFrame(), None)
        construction_directory = run_directory / pv_type
        construction_directory.mkdir(parents=True, exist_ok=True)
        demo = br.RadianceObj(name=f"{pv_type}_{run_id}", path=str(construction_directory), hpc=SETTINGS.hpc)
        demo.setGround(scenario.albedo)
        hour_starts, solar_times, metdata, timing = read_hourly_cultivation_weather(demo, scenario)
        weather = ALIGNED_HOURLY_WEATHER
        plan = make_sampling_plan(hour_starts, weather.GHI, weather.DHI, sampling_config)
        buckets = {key: group for key, group in plan.dropna(subset=["sample_hour"]).groupby("sample_hour")}
        if results is None:
            results = pd.DataFrame(index=hour_starts)
            results["hour_start"] = hour_starts.map(lambda t: t.isoformat())
            results["hour_end"] = (hour_starts + pd.Timedelta(hours=1)).map(lambda t: t.isoformat())
            results["duration_h"] = 1.0
            results["solar_timestamp"] = solar_times.map(lambda t: t.isoformat())
            results["sample_hour"] = plan.sample_hour.map(lambda t: t.isoformat() if pd.notna(t) else "").to_numpy()
            results["irradiance_scale"] = plan.scale.to_numpy()
            results["tracker_theta_deg"] = np.nan
        module = create_selected_module(demo, design, module_spec, pv_type, module_xgap, run_id)
        # pvlib tracker angles for every lit hour (cheap); scenes only for sampled hours.
        trackerdict = {} if metdata is None else demo.set1axis(
            metdata=metdata, azimuth=design.axis_azimuth, limit_angle=design.limit_angle,
            angledelta=None, backtrack=bool(design.backtrack), gcr=gcr, cumulativesky=False)
        columns = [f"{pv_type}_array_{face}_effective_incident_energy_Wh_hour" for face in ("front", "rear")]
        results[columns] = 0.0
        days = sorted(crop_dates if crop_dates is not None else set(hour_starts.date))
        ground_daily = {day: np.zeros((len(ground_points), 3)) for day in days}
        completed = 0
        spent = dict(sky=0., octree=0., pv=0., ground=0.)
        started = time.perf_counter()
        for timeindex, hour_end in enumerate([] if metdata is None else metdata.datetime):
            sample_hour = hour_end - pd.Timedelta(hours=1)
            if sample_hour not in buckets:
                continue
            bucket = buckets[sample_hour]
            hour_key = hour_end.strftime("%Y-%m-%d_%H%M")
            tick = time.perf_counter()
            skyfile = make_hourly_sky(demo, metdata, timeindex)
            spent["sky"] += time.perf_counter() - tick
            entry = trackerdict.get(hour_key)
            if entry is None:
                # No tracking angle (diffuse twilight): horizontal parking, as in Tianze's model.
                if metdata.dni[timeindex] > 0:
                    raise ValueError(f"Positive DNI but no tracker angle at {hour_key}.")
                entry = dict(surf_azm=(design.axis_azimuth - 90) % 360, surf_tilt=0.0, theta=0.0,
                             ghi=metdata.ghi[timeindex])
            if calculate_hao:
                results.loc[pd.DatetimeIndex(bucket.target_hour), "tracker_theta_deg"] = float(entry["theta"])
            tick = time.perf_counter()
            hourly_scene = demo.makeScene1axis(
                trackerdict={hour_key: entry.copy()}, module=module,
                sceneDict=dict(pitch=design.pitch, hub_height=design.hub_height,
                               nMods=array_spec.n_mods, nRows=array_spec.n_rows),
                cumulativesky=False)
            scene = hourly_scene[hour_key]["scenes"][0]
            octfile = demo.makeOct(filelist=[*demo.materialfiles, skyfile, *demo._getradfiles([scene])],
                                   octname=f"{pv_type}_{timeindex}_{run_id}")
            spent["octree"] += time.perf_counter() - tick
            if calculate_hao:
                tick = time.perf_counter()
                powers = np.asarray(analyze_tracking_array(pv_type, f"{run_id}_{timeindex}", demo.name, scene,
                                                           octfile, design, module_spec, array_spec))
                spent["pv"] += time.perf_counter() - tick
                results.loc[pd.DatetimeIndex(bucket.target_hour), columns] = bucket.scale.to_numpy()[:, None]*powers[None, :]
            crop_bucket = bucket if crop_dates is None else bucket[bucket.target_day.isin(crop_dates)]
            if calculate_linrui and not crop_bucket.empty:
                tick = time.perf_counter()
                direct, diffuse, total = analyze_ground_hour(octfile, demo.name, ground_scan,
                                                            f"ground_{pv_type}_{timeindex}_{run_id}",
                                                            with_components=ground_components)
                spent["ground"] += time.perf_counter() - tick
                values = np.column_stack([direct, diffuse, total])
                for day, weights in crop_bucket.groupby("target_day"):
                    ground_daily[day] += values*float(weights.scale.sum())
            completed += 1
            if completed % 20 == 0 or completed == len(buckets):
                elapsed = time.perf_counter() - started
                print(f"{pv_type} (single-axis): {completed}/{len(buckets)} sampled skies, {elapsed/60:.1f} min elapsed; "
                      "s/sky: " + ", ".join(f"{k} {v/completed:.1f}" for k, v in spent.items()), flush=True)
        if completed != len(buckets):
            raise RuntimeError("Some representative skies were absent from MetObj; do not accept incomplete reconstruction")
        if calculate_linrui:
            ground_summary[pv_type] = build_ground_day_frame(ground_daily, ground_points, pv_type,
                                                              with_components=ground_components)
        timing.update(sampled_skies_per_construction=completed, sampling_mode=sampling_config.mode,
                      reconstruction="sample irradiance ratio times target-hour GHI", mounting="single_axis",
                      derived_gcr=gcr)
    if results is None:
        raise ValueError("No construction selected")
    return results.reset_index(drop=True) if calculate_hao else pd.DataFrame(), ground_summary, timing


In [ ]:
# Cell 7c: Vertical bifacial physics (Tianze's vertical_pv.ipynb, adapted to sampling)
# Separate names from the fixed-tilt functions, so fixed-tilt code and cache stay unchanged.
# The scene is the fixed-tilt scene at tilt 90 deg; only the module sensors differ: they are
# built directly in float64 (bifacial_radiance's helper rounds to float16, which can swallow
# millimetre offsets of a vertical surface), one sensor per cell or equal opaque patch.


def validate_vertical_design(design: VerticalPVDesign, module_spec: ModuleSpecification) -> None:
    validate_cell_array_fit(design, module_spec)
    if design.pitch <= module_spec.module_thickness:
        raise ValueError("pitch must exceed the module thickness.")
    if design.clearance_height <= SETTINGS.ground_sensor_height:
        # Ground sensors would lie inside the panels (Tianze's collision check, applied to all rows).
        raise ValueError("Vertical clearance_height must exceed the ground sensor height "
                         f"({SETTINGS.ground_sensor_height} m).")


def make_vertical_rectangular_scans(scene, module_number, row_number, design, module_spec,
                                    first_x, first_y, spacing_x, spacing_y, surface_offset):
    azimuth = np.radians(round(design.azimuth, 2))
    x_axis = np.array([-np.cos(azimuth), np.sin(azimuth), 0.0])
    front_normal = np.array([np.sin(azimuth), np.cos(azimuth), 0.0])
    module = scene.module
    scene_dict = scene.sceneDict
    # Match the centring and height translation in bifacial_radiance's scene file.
    module_offset = (module_number - round(scene_dict["nMods"] / 1.999)) * module.scenex
    row_offset = (row_number - round(scene_dict["nRows"] / 1.999)) * design.pitch
    hub_height = design.clearance_height + module.sceney / 2 - module.offsetfromaxis
    scene_height = float(str(np.float16(hub_height)))
    centre = np.array([scene_dict.get("originx", 0.0), scene_dict.get("originy", 0.0), scene_height]) \
        + module_offset * x_axis - row_offset * front_normal
    first_centre = centre + first_x * x_axis + np.array([0.0, 0.0, first_y])
    scans = []
    # Rtrace's irradiance scan orientation points toward the receiving surface.
    for normal_offset, orientation in ((module.offsetfromaxis + module.z + surface_offset, -front_normal),
                                       (module.offsetfromaxis - surface_offset, front_normal)):
        point = first_centre + normal_offset * front_normal
        scans.append({
            "xstart": float(point[0]), "ystart": float(point[1]), "zstart": float(point[2]),
            "sx_xinc": float(spacing_x * x_axis[0]), "sx_yinc": float(spacing_x * x_axis[1]), "sx_zinc": 0.0,
            "xinc": 0.0, "yinc": 0.0, "zinc": spacing_y,
            "Nx": module_spec.numcellsx, "Ny": module_spec.numcellsy, "Nz": 1,
            "orient": " ".join(f"{value:.12g}" for value in orientation),
        })
    return scans[0], scans[1]


def make_vertical_surface_scans(pv_type, scene, module_number, row_number, design, module_spec):
    if pv_type == "semi_transparent":
        geometry = module_geometry(design, module_spec)
        spacing_x = module_spec.xcell + design.xcellgap
        spacing_y = module_spec.ycell + design.ycellgap
        centre_x = module_spec.xcell / 2 if module_spec.numcellsx % 2 == 0 else 0.0
        first_x = -geometry["grid_x"] / 2 + centre_x + module_spec.xcell / 2
        first_y = -geometry["grid_y"] / 2 + module_spec.ycell / 2
        surface_offset = module_spec.module_thickness / 2 + 0.001
        sample_area_m2 = module_spec.xcell * module_spec.ycell
    else:
        spacing_x = module_spec.module_x / module_spec.numcellsx
        spacing_y = module_spec.module_y / module_spec.numcellsy
        first_x = -module_spec.module_x / 2 + spacing_x / 2
        first_y = -module_spec.module_y / 2 + spacing_y / 2
        surface_offset = SETTINGS.conventional_surface_offset
        sample_area_m2 = spacing_x * spacing_y
    front_scan, back_scan = make_vertical_rectangular_scans(
        scene, module_number, row_number, design, module_spec,
        first_x, first_y, spacing_x, spacing_y, surface_offset)
    return front_scan, back_scan, sample_area_m2


def analyze_vertical_module(position, pv_type, run_id, simulation_name, scene, octfile, design, module_spec):
    row_number, module_number = position
    analysis = br.AnalysisObj(octfile, simulation_name, hpc=SETTINGS.hpc)
    front_scan, back_scan, sample_area_m2 = make_vertical_surface_scans(
        pv_type, scene, module_number, row_number, design, module_spec)
    name = f"{pv_type}_r{row_number:02d}_m{module_number:02d}_{run_id}"
    bifaciality = module_spec.bifaciality if pv_type == "semi_transparent" else module_spec.conventional_bifaciality
    if bifaciality > 0.0:
        front, back = analysis.analysis(octfile, name, front_scan, back_scan, accuracy=SETTINGS.accuracy)
        rear = float(np.asarray(back["Wm2"], dtype=float).sum() * sample_area_m2 * bifaciality)
    else:
        front = analysis.analysis(octfile, name, front_scan, accuracy=SETTINGS.accuracy)
        rear = 0.0
    return float(np.asarray(front["Wm2"], dtype=float).sum() * sample_area_m2), rear


def analyze_vertical_array(pv_type, run_id, simulation_name, scene, octfile, design, module_spec, array_spec):
    positions = [(r, m) for r in range(1, array_spec.n_rows + 1) for m in range(1, array_spec.n_mods + 1)]
    total_modules = len(positions)
    if SETTINGS.pv_trace_mode == "center":
        middle_row = (array_spec.n_rows + 1) // 2
        inner = [m for m in range(1, array_spec.n_mods + 1) if 1 < m < array_spec.n_mods]
        positions = [(middle_row, m) for m in (inner or range(1, array_spec.n_mods + 1))]
    elif SETTINGS.pv_trace_mode != "all":
        raise ValueError("SETTINGS.pv_trace_mode must be 'all' or 'center'.")
    scale = total_modules / len(positions)

    def run_module(position):
        return analyze_vertical_module(position, pv_type, run_id, simulation_name, scene, octfile, design, module_spec)

    front_total = rear_total = 0.0
    with ThreadPoolExecutor(max_workers=SETTINGS.n_workers) as executor:
        for front_power, rear_power in executor.map(run_module, positions):
            front_total += front_power
            rear_total += rear_power
    return front_total * scale, rear_total * scale


def calculate_hourly_results_vertical(
    design, scenario, module_spec, array_spec, run_id, run_directory,
    calculate_hao, calculate_linrui, ground_grid=None, sampling_config=None,
    crop_dates=None, constructions_to_run=None, ground_components=True,
):
    # Same interface, sampling and outputs as calculate_hourly_results (fixed tilt).
    import time
    from sampling_core import SamplingConfig, make_sampling_plan
    validate_vertical_design(design, module_spec)
    sampling_config = sampling_config or SamplingConfig(mode="full")
    crop_dates = set(crop_dates) if crop_dates is not None else None
    results, ground_summary, timing = None, {}, {}
    constructions = (("semi_transparent", SETTINGS.xgap_semi_transparent),
                     ("conventional", design.xgap_conventional))
    for pv_type, module_xgap in constructions:
        if constructions_to_run is not None and pv_type not in constructions_to_run:
            continue
        grid = ground_grid[pv_type] if isinstance(ground_grid, dict) else ground_grid
        if calculate_linrui and grid is None:
            raise ValueError("Vertical crop runs need the fixed plot grid from the engine")
        ground_points, ground_scan = (grid.points(), grid.scan()[0]) if grid is not None else (pd.DataFrame(), None)
        construction_directory = run_directory / pv_type
        construction_directory.mkdir(parents=True, exist_ok=True)
        demo = br.RadianceObj(name=f"{pv_type}_{run_id}", path=str(construction_directory), hpc=SETTINGS.hpc)
        demo.setGround(scenario.albedo)
        hour_starts, solar_times, metdata, timing = read_hourly_cultivation_weather(demo, scenario)
        weather = ALIGNED_HOURLY_WEATHER
        plan = make_sampling_plan(hour_starts, weather.GHI, weather.DHI, sampling_config)
        buckets = {key: group for key, group in plan.dropna(subset=["sample_hour"]).groupby("sample_hour")}
        if results is None:
            results = pd.DataFrame(index=hour_starts)
            results["hour_start"] = hour_starts.map(lambda t: t.isoformat())
            results["hour_end"] = (hour_starts + pd.Timedelta(hours=1)).map(lambda t: t.isoformat())
            results["duration_h"] = 1.0
            results["solar_timestamp"] = solar_times.map(lambda t: t.isoformat())
            results["sample_hour"] = plan.sample_hour.map(lambda t: t.isoformat() if pd.notna(t) else "").to_numpy()
            results["irradiance_scale"] = plan.scale.to_numpy()
        module = create_selected_module(demo, design, module_spec, pv_type, module_xgap, run_id)
        scene = demo.makeScene(module=module, sceneDict=dict(
            tilt=design.tilt, pitch=design.pitch, azimuth=design.azimuth,
            clearance_height=design.clearance_height, nMods=array_spec.n_mods, nRows=array_spec.n_rows),
            radname=f"{pv_type}_{run_id}")
        columns = [f"{pv_type}_array_{face}_effective_incident_energy_Wh_hour" for face in ("front", "rear")]
        results[columns] = 0.0
        days = sorted(crop_dates if crop_dates is not None else set(hour_starts.date))
        ground_daily = {day: np.zeros((len(ground_points), 3)) for day in days}
        completed = 0
        spent = dict(sky=0., octree=0., pv=0., ground=0.)
        started = time.perf_counter()
        for timeindex, hour_end in enumerate([] if metdata is None else metdata.datetime):
            sample_hour = hour_end - pd.Timedelta(hours=1)
            if sample_hour not in buckets:
                continue
            bucket = buckets[sample_hour]
            tick = time.perf_counter()
            skyfile = make_hourly_sky(demo, metdata, timeindex)
            spent["sky"] += time.perf_counter() - tick
            tick = time.perf_counter()
            octfile = demo.makeOct(filelist=[*demo.materialfiles, skyfile, *demo._getradfiles([scene])],
                                   octname=f"{pv_type}_{timeindex}_{run_id}")
            spent["octree"] += time.perf_counter() - tick
            if calculate_hao:
                tick = time.perf_counter()
                powers = np.asarray(analyze_vertical_array(pv_type, f"{run_id}_{timeindex}", demo.name, scene,
                                                           octfile, design, module_spec, array_spec))
                spent["pv"] += time.perf_counter() - tick
                results.loc[pd.DatetimeIndex(bucket.target_hour), columns] = bucket.scale.to_numpy()[:, None]*powers[None, :]
            crop_bucket = bucket if crop_dates is None else bucket[bucket.target_day.isin(crop_dates)]
            if calculate_linrui and not crop_bucket.empty:
                tick = time.perf_counter()
                direct, diffuse, total = analyze_ground_hour(octfile, demo.name, ground_scan,
                                                            f"ground_{pv_type}_{timeindex}_{run_id}",
                                                            with_components=ground_components)
                spent["ground"] += time.perf_counter() - tick
                values = np.column_stack([direct, diffuse, total])
                for day, weights in crop_bucket.groupby("target_day"):
                    ground_daily[day] += values*float(weights.scale.sum())
            completed += 1
            if completed % 20 == 0 or completed == len(buckets):
                elapsed = time.perf_counter() - started
                print(f"{pv_type} (vertical): {completed}/{len(buckets)} sampled skies, {elapsed/60:.1f} min elapsed; "
                      "s/sky: " + ", ".join(f"{k} {v/completed:.1f}" for k, v in spent.items()), flush=True)
        if completed != len(buckets):
            raise RuntimeError("Some representative skies were absent from MetObj; do not accept incomplete reconstruction")
        if calculate_linrui:
            ground_summary[pv_type] = build_ground_day_frame(ground_daily, ground_points, pv_type,
                                                              with_components=ground_components)
        timing.update(sampled_skies_per_construction=completed, sampling_mode=sampling_config.mode,
                      reconstruction="sample irradiance ratio times target-hour GHI", mounting="vertical")
    if results is None:
        raise ValueError("No construction selected")
    return results.reset_index(drop=True) if calculate_hao else pd.DataFrame(), ground_summary, timing


In [ ]:
# Cell 8: Persistent scenario records and selective temporary comparison

# Save only the four editable input groups: scenario, module, array, and design.
def build_input_record(
    design: PVDesign,
    scenario: CultivationScenario,
    module_spec: ModuleSpecification,
    array_spec: ArraySpecification,
) -> dict:
    return {
        # CultivationScenario
        "scenario_name": scenario.scenario_name,
        "crop_name": scenario.crop_name,
        "period_start": cultivation_period_bounds(scenario)[0].date().isoformat(),
        "period_end": cultivation_period_bounds(scenario)[1].date().isoformat(),
        "weather_filename": scenario.weather_filename,
        "albedo": scenario.albedo,
        "ground_grid_x": scenario.ground_grid_x,
        "ground_grid_y": scenario.ground_grid_y,
        # ModuleSpecification
        "module_name": module_spec.name,
        "module_x_m": module_spec.module_x,
        "module_y_m": module_spec.module_y,
        "module_thickness_m": module_spec.module_thickness,
        "numcellsx": module_spec.numcellsx,
        "numcellsy": module_spec.numcellsy,
        "xcell_m": module_spec.xcell,
        "ycell_m": module_spec.ycell,
        "semi_transparent_bifaciality": module_spec.bifaciality,
        "conventional_bifaciality": module_spec.conventional_bifaciality,
        "semi_transparent_glass_total_transmittance": module_spec.glass_total_transmittance,
        "semi_transparent_glass_specular_reflectance": module_spec.glass_specular_reflectance,
        "semi_transparent_glass_haze": module_spec.glass_haze,
        # ArraySpecification
        "array_n_mods": array_spec.n_mods,
        "array_n_rows": array_spec.n_rows,
        # PVDesign
        "tilt_deg": design.tilt,
        "pitch_m": design.pitch,
        "azimuth_deg": design.azimuth,
        "clearance_height_m": design.clearance_height,
        "xcellgap_m": design.xcellgap,
        "ycellgap_m": design.ycellgap,
        "xgap_conventional_m": design.xgap_conventional,
    }


# Append complete results without overwriting historical runs.
def append_result_rows(result_file: Path, new_rows: pd.DataFrame) -> None:
    result_file.parent.mkdir(parents=True, exist_ok=True)
    write_header = not result_file.exists() or result_file.stat().st_size == 0
    if not write_header:
        old_columns = list(pd.read_csv(result_file, nrows=0).columns)
        # Preserve historical columns and values if new input fields have been added.
        columns = list(dict.fromkeys([*old_columns, *new_rows.columns]))
        if columns != old_columns:
            previous = pd.read_csv(result_file).reindex(columns=columns)
            previous.to_csv(result_file, index=False, encoding="utf-8")
        new_rows = new_rows.reindex(columns=columns)
    new_rows.to_csv(result_file, mode="a", header=write_header, index=False, encoding="utf-8")


# Save only the final selected Linrui/Hao summaries; raw Radiance files remain temporary.
def save_selected_results(
    design, scenario, module_spec, array_spec,
    run_id, hao_hourly, ground_summary, calculate_linrui, calculate_hao, weather_timing,
) -> None:
    common_record = build_input_record(design, scenario, module_spec, array_spec)
    common_record.update(run_id=run_id, **weather_timing)
    if calculate_hao:
        hao_record = {key: value for key, value in common_record.items() if "glass_" not in key}
        rows = pd.DataFrame([hao_record] * len(hao_hourly))
        rows = pd.concat([rows, hao_hourly.reset_index(drop=True)], axis=1)
        append_result_rows(HAO_RESULTS_FILE, rows)
    if calculate_linrui:
        # ground_summary[pv_type] is now a tidy (day x point) DataFrame -- append
        # every row, tagged with this run's scenario/module/array/design fields
        # and its own pv_type/day/point_id, instead of one flattened scalar row.
        linrui_frames = []
        for pv_type, day_point_frame in ground_summary.items():
            rows = pd.DataFrame([common_record] * len(day_point_frame))
            rows = pd.concat([rows, day_point_frame.reset_index(drop=True)], axis=1)
            linrui_frames.append(rows)
        if linrui_frames:
            append_result_rows(LINRUI_RESULTS_FILE, pd.concat(linrui_frames, ignore_index=True))


# Calculate only the requested outputs, save summaries, then delete intermediates.
def model_for_optimization(
    design: PVDesign,
    scenario: CultivationScenario,
    module_spec: ModuleSpecification,
    array_spec: ArraySpecification,
    calculate_linrui: bool = True,
    calculate_hao: bool = True,
    ground_grid=None, sampling_config=None, crop_dates=None,
    constructions_to_run=None, persist_outputs=False, ground_components=True,
) -> tuple[pd.DataFrame, dict]:
    hao_hourly = pd.DataFrame()
    ground_summary = {}

    # With both switches off, avoid creating temporary scenes or result rows.
    if not calculate_linrui and not calculate_hao:
        return hao_hourly, ground_summary

    # Validate before creating a run ID, temporary files, Radiance scenes, or CSV rows.
    tracking = isinstance(design, TrackingDesign)
    vertical = isinstance(design, VerticalPVDesign)
    if tracking:
        validate_tracking_design(design, module_spec)
    if vertical:
        validate_vertical_design(design, module_spec)
    if (tracking or vertical) and persist_outputs:
        raise ValueError("persist_outputs is only implemented for fixed-tilt designs")
    validate_cell_array_fit(design, module_spec)
    for field in ("bifaciality", "conventional_bifaciality"):
        if not 0.0 <= getattr(module_spec, field) <= 1.0:
            raise ValueError(f"{field} must be between 0 and 1.")

    if calculate_linrui:
        validate_ground_grid(scenario)
    original_directory = Path.cwd().resolve()
    run_id = uuid4().hex[:8]
    with TemporaryDirectory(prefix=f"agripv_comparison_{run_id}_") as temporary_directory:
        try:
            simulate = (calculate_hourly_results_tracking if tracking else
                        calculate_hourly_results_vertical if vertical else calculate_hourly_results)
            hao_hourly, ground_summary, weather_timing = simulate(
                design, scenario, module_spec, array_spec, run_id,
                Path(temporary_directory) / "hourly", calculate_hao, calculate_linrui,
                ground_grid=ground_grid, sampling_config=sampling_config, crop_dates=crop_dates,
                constructions_to_run=constructions_to_run, ground_components=ground_components,
            )
        finally:
            # Leave RadianceObj's working directory before temporary cleanup.
            os.chdir(original_directory)

    if persist_outputs:
        save_selected_results(
        design, scenario, module_spec, array_spec,
        run_id, hao_hourly, ground_summary, calculate_linrui, calculate_hao, weather_timing,
    )
    return hao_hourly, ground_summary


In [ ]:
# Original independent demo disabled. Use Section D for controlled sampling.
RUN_TIANZE_DEMO = False


## Section A2 -- Fixed scenario/module/array config (must exist before Section B)

In [ ]:
# One shared annual weather calendar, including the complete crop season.
from dataclasses import replace

# FIELD SCALE (the same code runs both; only the size and therefore the run time differ).
# "small": proof of function. A fixed field 3 modules wide and 24 m deep. The number of rows
#          follows from the pitch: n_rows = floor(24 / pitch) = 6 rows at 4 m ... 2 rows at 12 m.
#          Only ONE module (middle row, middle module) is ray-traced for the electricity and its
#          result is multiplied by the number of modules; the other modules still shade it.
# "large": a wider and deeper field (10 modules wide, 60 m deep). Much slower; use for the final run.
# PV_TRACE_MODE below can still be set to "all" to trace every module (validation, very slow).
# Ground grid: keep the y spacing (field_depth / ground_grid_y) at about 0.5 m, far below the row pitch.
# 12 points over 24 m (2 m spacing) aliased against 4-5 m pitches and gave a wrong crop LER (test: 12 pts 0.730,
# 48 pts 0.827, 96 pts 0.824 for pitch 4.05 m).
# ======================= FIELD INPUTS (edit here) =======================
# The field size and the module layout are INPUTS of the model, not fixed assumptions.
# "small" is only a reduced field to save computing time; the delivered model accepts any field.
#   FIELD_N_MODS   : modules side by side in each row (the canopy width; "1 x 3" = 3)
#   FIELD_DEPTH_M  : depth of the field perpendicular to the rows [m]; n_rows = floor(depth / pitch)
FIELD_PRESET = "small"          # "small", "large" or "custom"
CUSTOM_N_MODS = 3               # used only when FIELD_PRESET = "custom"
CUSTOM_FIELD_DEPTH_M = 24.0     # used only when FIELD_PRESET = "custom"
GROUND_GRID_SPACING_M = 0.5     # custom field: ground points are placed every 0.5 m (see note above)
# =========================================================================
FIELD_PRESETS = {
    "small": dict(n_mods=3, field_depth_m=24., ground_grid_x=4, ground_grid_y=48, pv_trace_mode="center"),
    "large": dict(n_mods=10, field_depth_m=60., ground_grid_x=10, ground_grid_y=120, pv_trace_mode="center"),
}
if FIELD_PRESET == "custom":
    import math
    if int(CUSTOM_N_MODS) < 1 or float(CUSTOM_FIELD_DEPTH_M) <= 0:
        raise ValueError("CUSTOM_N_MODS must be >= 1 and CUSTOM_FIELD_DEPTH_M > 0")
    FIELD_PRESETS["custom"] = dict(
        n_mods=int(CUSTOM_N_MODS), field_depth_m=float(CUSTOM_FIELD_DEPTH_M),
        ground_grid_x=max(2, int(CUSTOM_N_MODS) + 1),
        ground_grid_y=max(12, math.ceil(float(CUSTOM_FIELD_DEPTH_M) / GROUND_GRID_SPACING_M)),
        pv_trace_mode="center")
if FIELD_PRESET not in FIELD_PRESETS:
    raise ValueError(f"FIELD_PRESET must be one of {list(FIELD_PRESETS)}")
_field = FIELD_PRESETS[FIELD_PRESET]
print("Field:", FIELD_PRESET, _field)
scenario = CultivationScenario(
    scenario_name="Aachen_Wheat_NSga2_v6", crop_name="Winter_wheat_103",
    start_date="2001-10-01", end_date="2002-09-30",
    weather_filename="Aachen_PVGIS_TMY.epw", weather_time_basis="pvgis_utc",
    ground_grid_x=_field["ground_grid_x"], ground_grid_y=_field["ground_grid_y"], albedo=.20,
)
# weather_time_basis="pvgis_utc": the EPW rows are UTC. Checked against solar geometry:
# read as UTC, GHI = DNI*cos(zenith)+DHI to 0.4 W/m2 on average and no hour has sun
# below the horizon; read as local time (UTC+1) the error is 22 W/m2 and 100 hours
# have beam sunlight at night. Radiance, crop weather and prices all use this same clock.
# The ground grid points are averaged by the crop run (area_mean_approximation), so a dense
# grid only adds ray-tracing cost. Final pointwise validation uses the same grid.
module_specification = ModuleSpecification(
    name="LONGi_LR5_72HGD_example", module_x=1.134, module_y=2.278,
    module_thickness=.030, numcellsx=6, numcellsy=24, xcell=.182, ycell=.091,
    bifaciality=.80, conventional_bifaciality=.80,
    glass_total_transmittance=.88, glass_specular_reflectance=.08, glass_haze=.20,
)
# n_rows is a placeholder: with field_depth_m > 0 it is derived from the pitch of every design.
array_specification = ArraySpecification(n_mods=_field["n_mods"], n_rows=1, field_depth_m=_field["field_depth_m"])
# PV ray-tracing scope. "all" = every module per sky (slow, reference behaviour).
# "center" = interior modules of the middle row, scaled to all modules (ONE module for n_mods=3).
# It ignores row-end and edge-row effects: check the difference against "all" before quoting
# absolute numbers. Rankings of designs were preserved in the 5x5 test (kWh -0.1 to -1.6 %).
PV_TRACE_MODE = _field["pv_trace_mode"]
SETTINGS = replace(SETTINGS, pv_trace_mode=PV_TRACE_MODE)


## Section B -- Linrui's real crop bridge (replaces RE.py's placeholder)

In [ ]:
# Crop interface: no expensive crop/radiation run during notebook setup.
import bridge
from weighted_engine import WeightedEngine, EngineConfig, load_price_profile, make_engine
from sampling_core import SamplingConfig, design_grid, compare_sampling
CROP_CONFIG = bridge.CropConfig(
    variety="Winter_wheat_103", start="2001-10-15", end="2002-08-31",
    start_type="sowing", accept_demo_assumptions=True, allow_missing_rain_for_pp=True,
)
# Soil/cultivar/calendar remain an uncalibrated illustration. PP has no water stress.


## Section C — NSGA-II design search and representative-hour weighting

In [ ]:
# LER + annual net income/ha; costs below are editable assumptions.
ENGINE_CONFIG = EngineConfig()
# Annual net subtracts capital recovery (25 years, 5% discount by default); it is not full NPV.
# Default costs from the literature (SYMBIOSYST D7.3, Bellone et al. 2024), see EngineConfig in
# weighted_engine.py; override e.g. EngineConfig(other_capex_eur_kwp=0.) to reproduce the v6 economics.
# Cell gaps remain fixed: fixed cell area means layout, NOT variable transparency.
FIXED_GAPS = dict(xcellgap=.0004, ycellgap=.00235, xgap_conventional=.20)
# With a fixed field the upper pitch bound must not exceed field_depth_m (search() checks this).
# ============================ MOUNTING TYPE ============================
# Set by the notebook: agripv_optimizer_fixed.ipynb = "fixed", agripv_optimizer_single_axis.ipynb = "single_axis",
# agripv_optimizer_vertical.ipynb = "vertical".
# Do not change it here; open the other notebook instead.
MOUNTING = "vertical"
# Module constructions to ray-trace and optimise. Both = the v6 behaviour (and v6 cache keys).
# ("conventional",) skips the slow semi-transparent tracing (~5-8x the conventional time),
# e.g. for a single-axis study within the time budget; it gets its own cache entries.
CONSTRUCTIONS = ("semi_transparent", "conventional")
# ==========================================================================
if MOUNTING == "fixed":
    DECISION_BOUNDS = {
        "tilt": (10., 50.), "pitch": (4., 12.),
        "azimuth": (120., 240.), "clearance_height": (2.5, 5.),
    }
    BASE_DESIGN = PVDesign(tilt=25., pitch=7., azimuth=180., clearance_height=3., **FIXED_GAPS)
elif MOUNTING == "single_axis":
    # pitch: row spacing; axis_azimuth: torque-tube bearing (180 = north-south axis, modules turn east-west);
    # hub_height: axis height; limit_angle: +/- rotation limit. Backtracking is fixed ON (not a decision).
    # The lowest module edge at the rotation limit must stay >= minimum_clearance_m (2.5 m):
    # with module_y = 2.278 m this needs hub_height >= about 2.5 + 1.14*sin(limit_angle).
    DECISION_BOUNDS = {
        "pitch": (4., 12.), "axis_azimuth": (120., 240.),
        "hub_height": (3., 5.5), "limit_angle": (30., 60.),
    }
    BASE_DESIGN = TrackingDesign(pitch=7., axis_azimuth=180., hub_height=4., limit_angle=45.,
                                 backtrack=True, **FIXED_GAPS)
elif MOUNTING == "vertical":
    # pitch: row spacing; azimuth: bearing of the FRONT face (90 = east, rows run north-south);
    # 0-180 covers every row orientation (180-360 only swaps front and rear face).
    # clearance_height: lower module edge; machinery drives BETWEEN the rows, so the 2.5 m
    # overhead clearance does not apply to vertical designs.
    DECISION_BOUNDS = {"pitch": (4., 12.), "azimuth": (0., 180.), "clearance_height": (.3, 1.5)}
    BASE_DESIGN = VerticalPVDesign(pitch=8., azimuth=90., clearance_height=.6, **FIXED_GAPS)
else:
    raise ValueError("MOUNTING must be 'fixed', 'single_axis' or 'vertical'")


## Section D -- Run settings (this is the cell you normally edit)

In [ ]:
# USER CONTROL PANEL
# FIELD SIZE / MODULES PER ROW are inputs too: see "FIELD INPUTS" at the top of Section A2 (FIELD_PRESET small/large/custom).
RUN_SEARCH = False  # True starts real Radiance + WOFOST. Setup alone does not run it.
POP_SIZE = 20        # NSGA-II population. First real run: 4 x 1. Budget run: 6 x 3.
N_GENERATIONS = 6   # raise only after the smoke test has given you a measured time per design
SEED = 1
# Run time (MEASURED on the i5-13600KF, 5x5 array, "center" tracing, 24 skies, 5 designs in parallel):
#   about 12 minutes of computing time per design (46-55 min for a batch of 5 at the same time).
# Time grows roughly linearly with the number of sampled skies (ESTIMATE):
#   combined 1 day/month x 2 hours/day  =  24 skies
#   combined 2 days/month x 3 hours/day =  72 skies  (about 3x)
#   combined 3 days/month x 4 hours/day = 144 skies  (about 6x)
# The "small" field (FIELD_PRESET in Section A2) should be faster; read the s/sky lines of the
# first run and update this estimate. Keep POP_SIZE a multiple of PARALLEL_DESIGNS.
# Vertical (east/west-facing) and single-axis designs need MORE hours per day than fixed tilt: few (near-noon) hours under-estimate the tracker
# and over-estimate fixed racks. Test (conventional, pitch 7 m): tracking gain vs fixed 35 deg was
# -15 % (1 h/day), -12 % (2), -1 % (4), +4 % (8 h/day) and had not fully converged. Use >= 8 h/day.
SAMPLING = SamplingConfig(mode="combined", hours_per_day=8, days_per_month=1)  # 96 skies

# Parallel designs: each design of one NSGA-II generation runs in its own process.
# SAVE this notebook before running (the workers re-read it). 1 = serial, as before.
# 5 designs at once worked well on the 20-thread i5-13600KF (each design uses 4 module threads).
# WORKERS_PER_DESIGN = module threads per design; None = (logical CPUs) // PARALLEL_DESIGNS.
PARALLEL_DESIGNS = 5
WORKERS_PER_DESIGN = None
NOTEBOOK_FILE = "agripv_optimizer_vertical.ipynb"
PRICE_MODE = "csv"  # uploaded 2020–2025 DE/LU tables; use flat_assumption only for sensitivity
FLAT_PRICE_EUR_KWH = .08
PRICE_YEARS = [2023, 2024, 2025]  # typical hourly profile averaged across these years (2020-2022 crisis years excluded)
# Uploaded tables use a complete 0-based UTC hour sequence; prices map to the fixed EPW standard clock.
RUN_FULL_VALIDATION = False
VALIDATE_TOP_N = 1  # best designs; full tracing, pointwise crop simulation (slow)
REFERENCE_PV_DESIGN = PVDesign(tilt=30., pitch=3., azimuth=180., clearance_height=.8,
                              xcellgap=.0004, ycellgap=.00235, xgap_conventional=.02)
# Reference is an explicit conventional ground-PV layout under identical weather/sampling.
# Check reference tilt/pitch/clearance for your thesis; no hardcoded kWh/ha denominator.

if RUN_SEARCH:
    engine = make_engine(globals(), CROP_CONFIG, ENGINE_CONFIG, SAMPLING,
                            REFERENCE_PV_DESIGN, PRICE_MODE, FLAT_PRICE_EUR_KWH, PRICE_YEARS,
                            parallel_designs=PARALLEL_DESIGNS, workers_per_design=WORKERS_PER_DESIGN,
                            notebook_path=NOTEBOOK_FILE, constructions=CONSTRUCTIONS)
    table = engine.search(POP_SIZE, N_GENERATIONS, SEED, DECISION_BOUNDS, BASE_DESIGN)
    pareto = table[table["pareto"] & table["feasible"]].copy()
    engine.export(table)
    display(pareto)
    if RUN_FULL_VALIDATION:
        validation = engine.validate_finalists(pareto, VALIDATE_TOP_N)
        display(validation)
else:
    engine, table, pareto = None, None, None
    print("Setup complete. Edit Section D; set RUN_SEARCH=True for real NSGA-II + Radiance/WOFOST runs.")


## Section E -- Results

In [ ]:
# The Pareto set is a vector of non-dominated feasible trade-offs, not one weighted winner.
if table is not None:
    display(table.loc[table.pareto & table.feasible, [
        "design_id", "pv_type", "LER", "annual_net_eur_ha", "pareto"
    ]])
    print("Outputs: results/" + ("" if MOUNTING == "fixed" else MOUNTING + "_") + "nsga2_<sampling_mode>/pareto_front.csv")
    print("For a single recommendation, choose transparently from the Pareto set after reviewing the trade-off.")


## Section F -- Stage 2: re-check the Pareto designs with a denser sampling

Run after the search. The same designs are evaluated again with more sampled skies (default: 3 days/month x 4 hours/day = 144 skies); quote the numbers of this table in the thesis. Results go to `results/recheck_<mode>_<n>skies/` (evaluated_candidates.csv, pareto_front.csv, comparison_with_search.csv).

In [ ]:
# Stage 2 control panel (separate from Section D; the model workers never execute this cell).
RUN_RECHECK = False
RECHECK_SAMPLING = SamplingConfig(mode="combined", hours_per_day=8, days_per_month=2)  # 192 skies per construction
# None = Pareto table of this session. After a kernel restart use a file instead, e.g.
# RECHECK_SOURCE = r"C:\Users\haoli\Agripv\optimizer\results\nsga2_combined\pareto_front.csv"
RECHECK_SOURCE = None

if RUN_RECHECK:
    if RECHECK_SOURCE:
        recheck_source = pd.read_csv(RECHECK_SOURCE, float_precision="round_trip")
    elif pareto is not None:
        recheck_source = pareto
    else:
        # Kernel restarted or search not run in this session: use the newest saved Pareto file.
        prefix = "" if MOUNTING == "fixed" else MOUNTING + "_"
        saved = sorted(Path("results").glob(prefix + "nsga2_*/pareto_front.csv"), key=lambda f: f.stat().st_mtime)
        if not saved:
            raise ValueError("No Pareto table found: run the search first, or set RECHECK_SOURCE to a pareto_front.csv")
        print("Using saved Pareto file:", saved[-1])
        recheck_source = pd.read_csv(saved[-1], float_precision="round_trip")
    if engine is None:
        engine = make_engine(globals(), CROP_CONFIG, ENGINE_CONFIG, SAMPLING, REFERENCE_PV_DESIGN,
                                PRICE_MODE, FLAT_PRICE_EUR_KWH, PRICE_YEARS,
                                parallel_designs=PARALLEL_DESIGNS, workers_per_design=WORKERS_PER_DESIGN,
                                notebook_path=NOTEBOOK_FILE, constructions=CONSTRUCTIONS)
    recheck_table = engine.recheck(recheck_source, RECHECK_SAMPLING)
    display(recheck_table.loc[recheck_table.pareto & recheck_table.feasible,
                              ["design_id", "pv_type", "n_rows", "LER", "annual_net_eur_ha", "n_skies"]])
    print("Comparison with the search: results/recheck_<mode>_<n>skies/comparison_with_search.csv")